In [6]:
import sqlite3
import pandas as pd

# 1. Definieer de bestandsnamen
db_naam = 'BikeToDrive.db'
sql_script = 'schema.sql'

# 2. Maak verbinding met de SQLite database (maakt het bestand aan als het niet bestaat)
conn = sqlite3.connect(db_naam)
cursor = conn.cursor()

# 3. Lees het SQL-script en voer het uit
print("Datamodel aanmaken...")
with open(sql_script, 'r') as file:
    sql_queries = file.read()
    cursor.executescript(sql_queries)

print("Datamodel succesvol aangemaakt!")


conn.commit()
conn.close()

Datamodel aanmaken...


OperationalError: table Leverancier already exists

In [ ]:
#function to make the database empty
def reset_database():
    conn = sqlite3.connect(db_naam)
    cursor = conn.cursor()
    
    # List of tables to clear
    tables = [ 'Fiets_Verkoop', 
        'Accessoire_Verkoop', 
        'Fiets_Inkoop', 
        'Accessoire_Inkoop', 
        'Onderhoud',
        'Fiets', 
        'Accessoire', 
        'Monteur', 
        'Leverancier', 
        'Fabrikant', 
        'Klant', 
        'Filiaal']  # Voeg hier alle tabellen toe die je wilt leegmaken
    
    for table in tables:
        cursor.execute(f'DELETE FROM {table}')
    
    conn.commit()
    conn.close()
    print("Database is gereset (leeg gemaakt).")

In [10]:
import sqlite3
import pandas as pd

# 1. Connecties opzetten
conn_acc_verkoop = sqlite3.connect("C:\\Users\\xande\\School\\DEAI - WC Source Data Model Implementation\\BikeToDrive_1_Accessoireverkoop.db")
conn_fiets_verkoop = sqlite3.connect('C:\\Users\\xande\\School\\DEAI - WC Source Data Model Implementation\\BikeToDrive_2_Fietsverkoop.db')
conn_onderhoud = sqlite3.connect('C:\\Users\\xande\\School\\DEAI - WC Source Data Model Implementation\\BikeToDrive_3_Onderhoud.db')
conn_acc_inkoop = sqlite3.connect('C:\\Users\\xande\\School\\DEAI - WC Source Data Model Implementation\\BikeToDrive_4_Accessoire_Inkoop.db')
conn_fiets_inkoop = sqlite3.connect('C:\\Users\\xande\\School\\DEAI - WC Source Data Model Implementation\\BikeToDrive_5_Fiets_Inkoop.db')

db_naam = 'BikeToDrive.db'
conn = sqlite3.connect(db_naam)

# 2. Mapping maken van connecties naar de tabellen die ze BEVATTEN
db_mapping = {
    conn_acc_inkoop: ['Leverancier', 'Accessoire', 'Accessoire_Inkoop'],
    conn_fiets_inkoop: ['Fabrikant', 'Fiets', 'Fiets_Inkoop'],
    conn_onderhoud: ['Onderhoud', 'Monteur', 'Filiaal', 'Fiets', 'Fabrikant'],
    conn_acc_verkoop: ['Accessoire', 'Accessoire_Verkoop', 'Klant', 'Leverancier', 'Filiaal', 'Monteur'],
    conn_fiets_verkoop: ['Fiets', 'Fiets_Verkoop', 'Klant', 'Fabrikant', 'Filiaal', 'Monteur']
}

# 3. Data verzamelen en inladen
def load_and_merge_data(mapping, target_conn):
    # We slaan alle dataframes tijdelijk op per tabelnaam om duplicates eruit te halen
    table_data = {}

    # Lees alle tabellen uit de juiste databases
    for source_conn, tables in mapping.items():
        for table in tables:
            try:
                df = pd.read_sql_query(f"SELECT * FROM {table}", source_conn)
                
                # Als we de tabel al eerder hebben gezien (uit een andere DB), plak hem er dan onder
                if table in table_data:
                    table_data[table] = pd.concat([table_data[table], df], ignore_index=True)
                else:
                    table_data[table] = df
            except Exception as e:
                print(f"Fout bij het laden van {table}: {e}")

    # Schrijf alles naar de nieuwe target database (en verwijder dubbele rijen)
    for table_name, combined_df in table_data.items():
        # Verwijder exacte kopieën van rijen (belangrijk voor Klant, Filiaal, etc.)
        deduplicated_df = combined_df.drop_duplicates()
        
        # Schrijf naar de database (replace is hier veilig omdat we alle data al in het geheugen gecombineerd hebben)
        deduplicated_df.to_sql(table_name, target_conn, if_exists='replace', index=False)
        print(f"✅ Tabel '{table_name}' succesvol geladen met {len(deduplicated_df)} rijen.")

# Voer de functie uit
load_and_merge_data(db_mapping, conn)

# Connecties sluiten is een goede gewoonte!
conn_acc_verkoop.close()
conn_fiets_verkoop.close()
conn_onderhoud.close()
conn_acc_inkoop.close()
conn_fiets_inkoop.close()
conn.close()


✅ Tabel 'Leverancier' succesvol geladen met 10 rijen.
✅ Tabel 'Accessoire' succesvol geladen met 13 rijen.
✅ Tabel 'Accessoire_Inkoop' succesvol geladen met 50 rijen.
✅ Tabel 'Fabrikant' succesvol geladen met 31 rijen.
✅ Tabel 'Fiets' succesvol geladen met 105 rijen.
✅ Tabel 'Fiets_Inkoop' succesvol geladen met 100 rijen.
✅ Tabel 'Onderhoud' succesvol geladen met 50 rijen.
✅ Tabel 'Monteur' succesvol geladen met 16 rijen.
✅ Tabel 'Filiaal' succesvol geladen met 13 rijen.
✅ Tabel 'Accessoire_Verkoop' succesvol geladen met 100 rijen.
✅ Tabel 'Klant' succesvol geladen met 45 rijen.
✅ Tabel 'Fiets_Verkoop' succesvol geladen met 150 rijen.
